# 03 — Secrets et tokens

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer la différence entre `random` et `secrets` ;
- générer des tokens, URL-safe strings et identifiants sécurisés ;
- utiliser `os.urandom()` et `secrets` pour la cryptographie ;
- implémenter des tokens à usage unique et à expiration ;
- appliquer les bonnes pratiques pour la gestion des secrets.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le hachage (`hashlib`, `hmac`) — notebook 01 ;
- le hachage de mots de passe (`argon2`) — notebook 02 ;
- le module `random` pour les usages non-sécuritaires.

## Plan

1. `random` vs `secrets` — pourquoi ça compte
2. Le module `secrets`
3. `os.urandom()` — la source d'entropie
4. Générer des tokens d'API
5. Tokens à usage unique (OTP)
6. Tokens à expiration
7. Gestion des secrets applicatifs
8. Synthèse
9. Exercices
10. Ressources

---

## 1. `random` vs `secrets` — pourquoi ça compte

In [ ]:
import random
import secrets

| Critère | `random` | `secrets` |
|---|---|---|
| Algorithme | Mersenne Twister (PRNG) | CSPRNG (OS) |
| Déterministe | Oui (seed) | Non |
| Prédictible | **Oui** (après ~624 valeurs observées) | Non |
| Usage | Simulations, jeux, tests | Sécurité, tokens, clés |

### Démonstration : `random` est prédictible

In [ ]:
# random utilise un seed déterministe
random.seed(42)
valeurs_1 = [random.randint(0, 100) for _ in range(5)]

random.seed(42)
valeurs_2 = [random.randint(0, 100) for _ in range(5)]

print(f"Série 1 : {valeurs_1}")
print(f"Série 2 : {valeurs_2}")
print(f"Identiques : {valeurs_1 == valeurs_2}")

In [ ]:
# secrets n'est pas reproductible
valeurs_s1 = [secrets.randbelow(100) for _ in range(5)]
valeurs_s2 = [secrets.randbelow(100) for _ in range(5)]
print(f"Série 1 : {valeurs_s1}")
print(f"Série 2 : {valeurs_s2}")
print(f"Identiques : {valeurs_s1 == valeurs_s2}")

**Règle absolue :** utilisez `secrets` (ou `os.urandom()`) pour **tout ce qui touche à la sécurité** : tokens, clés, sels, identifiants de session, codes de vérification.

---

## 2. Le module `secrets`

`secrets` (Python 3.6+) fournit des fonctions de haut niveau pour générer des valeurs aléatoires cryptographiquement sûres.

### `secrets.token_bytes` — octets bruts

In [ ]:
# 32 octets (256 bits) de données aléatoires
token = secrets.token_bytes(32)
print(f"token_bytes : {token.hex()}")
print(f"Longueur : {len(token)} octets")

### `secrets.token_hex` — chaîne hexadécimale

In [ ]:
# 32 octets → 64 caractères hex
token = secrets.token_hex(32)
print(f"token_hex : {token}")
print(f"Longueur : {len(token)} caractères")

### `secrets.token_urlsafe` — URL-safe Base64

In [ ]:
# Idéal pour les URLs (pas de caractères spéciaux)
token = secrets.token_urlsafe(32)
print(f"token_urlsafe : {token}")
print(f"Longueur : {len(token)} caractères")

### `secrets.randbelow` — entier aléatoire

In [ ]:
# Entier aléatoire dans [0, n)
print(f"Dé à 6 faces : {secrets.randbelow(6) + 1}")
print(f"Code PIN 6 chiffres : {secrets.randbelow(1_000_000):06d}")

### `secrets.choice` — choix aléatoire

In [ ]:
import string

# Générer un mot de passe aléatoire
alphabet = string.ascii_letters + string.digits + "!@#$%^&*"
mot_de_passe = "".join(secrets.choice(alphabet) for _ in range(16))
print(f"Mot de passe : {mot_de_passe}")

### Taille recommandée

| Usage | Taille minimale | `secrets` call |
|---|---|---|
| Token d'API | 32 octets (256 bits) | `token_urlsafe(32)` |
| Session ID | 32 octets | `token_hex(32)` |
| Clé de chiffrement | 32 octets (AES-256) | `token_bytes(32)` |
| Code de confirmation email | 32 octets | `token_urlsafe(32)` |
| Code PIN temporaire | 6-8 chiffres | `randbelow(10**6)` |

---

## 3. `os.urandom()` — la source d'entropie

`secrets` utilise `os.urandom()` en interne. Cette fonction lit directement le générateur de nombres aléatoires du système d'exploitation.

In [ ]:
import os

# Équivalent bas-niveau de secrets.token_bytes
octets = os.urandom(32)
print(f"os.urandom : {octets.hex()}")

| OS | Source |
|---|---|
| Linux | `/dev/urandom` (alimenté par l'entropie du noyau) |
| macOS | `SecRandomCopyBytes` |
| Windows | `CryptGenRandom` |

Toutes ces sources sont des **CSPRNG** (Cryptographically Secure Pseudo-Random Number Generator).

### `random.SystemRandom` — pont entre `random` et CSPRNG

In [ ]:
# Si vous avez besoin de l'API random mais avec du CSPRNG
sysrand = random.SystemRandom()
print(f"randint : {sysrand.randint(1, 100)}")
print(f"choice  : {sysrand.choice(['a', 'b', 'c', 'd'])}")
print(f"uniform : {sysrand.uniform(0.0, 1.0):.6f}")

---

## 4. Générer des tokens d'API

In [ ]:
def generer_api_key(prefix: str = "sk") -> str:
    """Génère une clé API au format prefix_<token>."""
    return f"{prefix}_{secrets.token_urlsafe(32)}"

cle = generer_api_key()
print(f"Clé API : {cle}")

### Stocker le hash, pas la clé

In [ ]:
import hashlib

def creer_api_key():
    """Retourne (clé_pour_utilisateur, hash_pour_bdd)."""
    cle = f"sk_{secrets.token_urlsafe(32)}"
    hash_cle = hashlib.sha256(cle.encode()).hexdigest()
    return cle, hash_cle

def verifier_api_key(cle_recue: str, hash_stocke: str) -> bool:
    """Vérifie une clé API contre son hash stocké."""
    import hmac as hmac_mod
    hash_recue = hashlib.sha256(cle_recue.encode()).hexdigest()
    return hmac_mod.compare_digest(hash_recue, hash_stocke)

# Création
cle, hash_cle = creer_api_key()
print(f"Clé (montrée une seule fois) : {cle}")
print(f"Hash (stocké en BDD)         : {hash_cle}")

# Vérification
print(f"Valide : {verifier_api_key(cle, hash_cle)}")
print(f"Fausse : {verifier_api_key('sk_fausse_cle', hash_cle)}")

**Pourquoi hasher la clé API ?** Si la base de données est compromise, l'attaquant n'obtient que les hashes — il ne peut pas utiliser les clés.

---

## 5. Tokens à usage unique (OTP)

In [ ]:
import time
import hashlib

class TokenStore:
    """Stockage de tokens à usage unique."""

    def __init__(self):
        self._tokens = {}  # hash → (données, expiration)

    def creer(self, donnees: str, duree_sec: int = 3600) -> str:
        token = secrets.token_urlsafe(32)
        hash_token = hashlib.sha256(token.encode()).hexdigest()
        self._tokens[hash_token] = (donnees, time.monotonic() + duree_sec)
        return token

    def utiliser(self, token: str) -> str | None:
        hash_token = hashlib.sha256(token.encode()).hexdigest()
        if hash_token not in self._tokens:
            return None
        donnees, expiration = self._tokens.pop(hash_token)  # usage unique
        if time.monotonic() > expiration:
            return None
        return donnees

In [ ]:
store = TokenStore()

# Créer un token de confirmation d'email
token = store.creer("user_42@example.com", duree_sec=3600)
print(f"Token envoyé par email : {token}")

# L'utilisateur clique sur le lien
email = store.utiliser(token)
print(f"Email confirmé : {email}")

# Deuxième utilisation → None (usage unique)
print(f"Réutilisation : {store.utiliser(token)}")

---

## 6. Tokens à expiration (signés)

Plutôt que de stocker les tokens côté serveur, on peut les **signer** avec HMAC. Le serveur vérifie la signature sans base de données.

In [ ]:
import hmac as hmac_mod
import base64
import json as json_mod

class SignedToken:
    def __init__(self, secret: bytes):
        self.secret = secret

    def creer(self, payload: dict, duree_sec: int = 3600) -> str:
        payload["exp"] = int(time.time()) + duree_sec
        body = base64.urlsafe_b64encode(
            json_mod.dumps(payload).encode()
        ).decode()
        sig = hmac_mod.new(self.secret, body.encode(), hashlib.sha256).hexdigest()
        return f"{body}.{sig}"

    def verifier(self, token: str) -> dict | None:
        try:
            body, sig = token.rsplit(".", 1)
        except ValueError:
            return None
        sig_attendue = hmac_mod.new(self.secret, body.encode(), hashlib.sha256).hexdigest()
        if not hmac_mod.compare_digest(sig, sig_attendue):
            return None
        payload = json_mod.loads(base64.urlsafe_b64decode(body))
        if time.time() > payload.get("exp", 0):
            return None
        return payload

In [ ]:
signer = SignedToken(secret=secrets.token_bytes(32))

token = signer.creer({"user_id": 42, "role": "admin"}, duree_sec=3600)
print(f"Token signé : {token[:60]}...")

payload = signer.verifier(token)
print(f"Payload : {payload}")

# Token altéré
print(f"Altéré : {signer.verifier(token + 'x')}")

Ce pattern est exactement ce que fait **JWT** (JSON Web Token) en version simplifiée. En production, utilisez `PyJWT` ou `python-jose`.

---

## 7. Gestion des secrets applicatifs

### Où stocker les secrets ?

| Méthode | Sécurité | Adapté pour |
|---|---|---|
| Variable d'environnement | Moyenne | Dev / CI/CD |
| Fichier `.env` (non versionné) | Moyenne | Dev local |
| Vault (HashiCorp, AWS SM, GCP SM) | Élevée | Production |
| Keyring OS | Bonne | Applications desktop |

**Ne versionnez JAMAIS un secret dans git.**

### Lire depuis l'environnement

In [ ]:
import os

# Pattern recommandé : lever une erreur si le secret est absent
def get_secret(nom: str) -> str:
    val = os.environ.get(nom)
    if val is None:
        raise RuntimeError(f"Variable d'environnement {nom!r} non définie")
    return val

# Exemple (ne pas exécuter si la variable n'existe pas)
try:
    cle = get_secret("API_SECRET_KEY")
    print(f"Clé chargée ({len(cle)} caractères)")
except RuntimeError as e:
    print(f"Attendu : {e}")

### Utiliser `keyring` pour le stockage sécurisé

In [ ]:
try:
    import keyring

    # Stocker un secret
    # keyring.set_password("mon_app", "api_key", "secret_value")

    # Récupérer un secret
    # val = keyring.get_password("mon_app", "api_key")

    print("keyring est disponible")
    print(f"Backend : {keyring.get_keyring()}")
except ImportError:
    print("keyring non installé — pip install keyring")

---

## 8. Synthèse

| Fonction | Usage |
|---|---|
| `secrets.token_urlsafe(n)` | Token pour URLs et API |
| `secrets.token_hex(n)` | Token hexadécimal |
| `secrets.token_bytes(n)` | Octets aléatoires bruts |
| `secrets.randbelow(n)` | Entier aléatoire sécurisé |
| `secrets.choice(seq)` | Choix aléatoire sécurisé |
| `os.urandom(n)` | Source d'entropie bas-niveau |

**Règles à retenir :**
- `random` pour les simulations, `secrets` pour la sécurité. **Pas de mélange.**
- Minimum 32 octets (256 bits) pour les tokens de sécurité.
- Stockez le **hash** des tokens d'API, pas les tokens en clair.
- Les tokens à usage unique doivent être supprimés après utilisation.
- Ne versionnez **jamais** de secrets dans git.

---

## 9. Exercices

### Exercice 1 — Générateur de mots de passe *(facile)*

Écrire une fonction `generer_mdp(longueur: int = 16) -> str` qui génère un mot de passe aléatoire contenant au moins une majuscule, une minuscule, un chiffre et un caractère spécial.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Secrets_et_tokens", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import secrets
import string

def generer_mdp(longueur: int = 16) -> str:
    alphabet = string.ascii_letters + string.digits + "!@#$%^&*()-_=+"
    while True:
        mdp = "".join(secrets.choice(alphabet) for _ in range(longueur))
        # Vérifier les contraintes
        if (any(c in string.ascii_lowercase for c in mdp) and
            any(c in string.ascii_uppercase for c in mdp) and
            any(c in string.digits for c in mdp) and
            any(c in "!@#$%^&*()-_=+" for c in mdp)):
            return mdp

for _ in range(5):
    print(generer_mdp())
```

</details>

### Exercice 2 — Token d'invitation *(moyen)*

Implémenter un système de tokens d'invitation :
- `creer_invitation(email, role, duree_jours)` → token
- `utiliser_invitation(token)` → dict ou None (usage unique + expiration)
- Les tokens sont stockés hashés.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Secrets_et_tokens", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import secrets
import hashlib
import time

class InvitationService:
    def __init__(self):
        self._invitations = {}

    def creer_invitation(self, email: str, role: str, duree_jours: int = 7) -> str:
        token = secrets.token_urlsafe(32)
        h = hashlib.sha256(token.encode()).hexdigest()
        self._invitations[h] = {
            "email": email,
            "role": role,
            "expire": time.time() + duree_jours * 86400,
        }
        return token

    def utiliser_invitation(self, token: str) -> dict | None:
        h = hashlib.sha256(token.encode()).hexdigest()
        invitation = self._invitations.pop(h, None)
        if invitation is None:
            return None
        if time.time() > invitation["expire"]:
            return None
        return invitation

svc = InvitationService()
tok = svc.creer_invitation("bob@example.com", "editor", duree_jours=7)
print(f"Token : {tok}")
print(f"Utiliser : {svc.utiliser_invitation(tok)}")
print(f"Réutiliser : {svc.utiliser_invitation(tok)}")
```

</details>

### Exercice 3 — Comparaison timing attack *(moyen)*

Démontrer qu'une comparaison `==` fuit de l'information temporelle :
1. Générer un secret de 32 caractères hex ;
2. Comparer avec des candidats partageant 0, 8, 16, 24, 32 caractères corrects ;
3. Mesurer le temps avec `time.perf_counter_ns()` sur 100 000 itérations ;
4. Montrer que `hmac.compare_digest` ne présente pas cette fuite.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Secrets_et_tokens", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import secrets
import hmac
import time

secret = secrets.token_hex(16)  # 32 caractères hex

def mesurer(fn, n=100_000):
    t0 = time.perf_counter_ns()
    for _ in range(n):
        fn()
    return (time.perf_counter_ns() - t0) / n

# Candidats avec 0, 8, 16, 24, 32 caractères corrects
for nb_correct in [0, 8, 16, 24, 32]:
    candidat = secret[:nb_correct] + "0" * (32 - nb_correct)

    t_eq = mesurer(lambda: candidat == secret)
    t_hmac = mesurer(lambda: hmac.compare_digest(candidat, secret))
    print(f"Correct: {nb_correct:2d}/32  ==: {t_eq:.0f} ns  compare_digest: {t_hmac:.0f} ns")

# == montre une tendance (plus long quand plus de caractères corrects)
# compare_digest reste constant
```

</details>

### Exercice 4 — Mini-JWT *(difficile)*

Implémenter un mini-JWT complet avec :
- `encode(payload, secret, duree_sec)` → token string
- `decode(token, secret)` → payload ou raise `TokenError`
- Support de l'expiration (`exp`) et du `not before` (`nbf`)
- Header avec `alg: HS256`

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Secrets_et_tokens", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import base64
import hashlib
import hmac
import json
import time

class TokenError(Exception):
    pass

def _b64url_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode()

def _b64url_decode(s: str) -> bytes:
    padding = 4 - len(s) % 4
    return base64.urlsafe_b64decode(s + "=" * padding)

def encode(payload: dict, secret: str, duree_sec: int = 3600) -> str:
    header = {"alg": "HS256", "typ": "JWT"}
    payload = {**payload, "exp": int(time.time()) + duree_sec, "iat": int(time.time())}

    h = _b64url_encode(json.dumps(header).encode())
    p = _b64url_encode(json.dumps(payload).encode())
    message = f"{h}.{p}"
    sig = hmac.new(secret.encode(), message.encode(), hashlib.sha256).digest()
    return f"{message}.{_b64url_encode(sig)}"

def decode(token: str, secret: str) -> dict:
    parts = token.split(".")
    if len(parts) != 3:
        raise TokenError("Format invalide")
    h, p, s = parts
    sig_attendue = hmac.new(secret.encode(), f"{h}.{p}".encode(), hashlib.sha256).digest()
    sig_recue = _b64url_decode(s)
    if not hmac.compare_digest(sig_attendue, sig_recue):
        raise TokenError("Signature invalide")
    payload = json.loads(_b64url_decode(p))
    now = time.time()
    if now > payload.get("exp", float("inf")):
        raise TokenError("Token expiré")
    if now < payload.get("nbf", 0):
        raise TokenError("Token pas encore valide")
    return payload

secret = "mon-secret-jwt"
token = encode({"user": "alice", "role": "admin"}, secret, duree_sec=3600)
print(f"Token : {token[:50]}...")
print(f"Payload : {decode(token, secret)}")

try:
    decode(token, "mauvais-secret")
except TokenError as e:
    print(f"Erreur attendue : {e}")
```

</details>

---

## 10. Ressources

- [Module `secrets` — documentation officielle](https://docs.python.org/3/library/secrets.html)
- [PEP 506 — Adding A Secrets Module](https://peps.python.org/pep-0506/)
- [OWASP — Session Management Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Session_Management_Cheat_Sheet.html)
- [PyJWT — JSON Web Tokens](https://pyjwt.readthedocs.io/)